# EMIT O₂ A-Band Analysis

This notebook analyzes **NASA EMIT Level 1B at-sensor radiance** over the Republic of Georgia (3 June 2026) to examine the atmospheric O₂ A-band near 760 nm and its relationship with surface elevation.

The analysis produces four portfolio figures:

1. L1B radiance spectrum across the O₂ A-band
2. Spatial map of the 760/775 nm O₂ index
3. Continuum-normalized O₂ A-band depth versus surface elevation
4. Spectral dependence of elevation sensitivity

**Interpretation note:** this is a proof-of-concept analysis of O₂ A-band sensitivity in real orbital radiance. It does **not** demonstrate geometric range retrieval from orbit.


## 1. Imports and configuration

The raw EMIT `.nc` file is intentionally **not stored in this repository**.  
To run the notebook, place the L1B file either:

- in a local `data/` directory next to the notebook, or
- directly next to the notebook.

Alternatively, set the `EMIT_L1B_FILE` environment variable outside the notebook.


In [ ]:
from pathlib import Path
import os

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# -----------------------------
# Plot / output configuration
# -----------------------------

OUT_DIR = Path("figures")
OUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# -----------------------------
# Source product
# -----------------------------

L1B_FILENAME = "EMIT_L1B_RAD_001_20260603T075933_2615405_006.nc"


def find_data_file(filename, env_var=None):
    """Return a local data file without hard-coding a personal computer path."""

    candidates = []

    if env_var and os.environ.get(env_var):
        candidates.append(Path(os.environ[env_var]))

    candidates.extend([
        Path("data") / filename,
        Path(filename),
    ])

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    locations = "\n".join(f"  - {p}" for p in candidates)

    raise FileNotFoundError(
        f"Could not find {filename}.\n"
        "Place the file in data/, place it next to this notebook, "
        f"or set {env_var}.\n\nChecked:\n{locations}"
    )


L1B_FILE = find_data_file(L1B_FILENAME, "EMIT_L1B_FILE")

print("Using:", L1B_FILE.name)


## 2. Open EMIT radiance, wavelength metadata, and geolocation

In [ ]:
rad = xr.open_dataset(L1B_FILE)
sensor = xr.open_dataset(L1B_FILE, group="sensor_band_parameters")
loc = xr.open_dataset(L1B_FILE, group="location")

wl = sensor["wavelengths"].values
fwhm = sensor["fwhm"].values
good_wavelengths = sensor["good_wavelengths"].values
elev = loc["elev"].values

print(rad)
print("\nNumber of spectral bands:", len(wl))
print("Scene elevation range:",
      float(np.nanmin(elev)), "to", float(np.nanmax(elev)), "m")


## 3. Inspect the O₂ A-band channels

In [ ]:
region = (wl >= 730) & (wl <= 800)

print("EMIT channels from 730–800 nm")
print(" band   wavelength_nm   FWHM_nm   good")

for i in np.where(region)[0]:
    print(
        f"{i + 1:5d}"
        f"{wl[i]:16.3f}"
        f"{fwhm[i]:10.3f}"
        f"{good_wavelengths[i]:7.0f}"
    )


def nearest_band(target_nm):
    """Index of the EMIT channel nearest a requested wavelength."""
    return int(np.argmin(np.abs(wl - target_nm)))


i753 = nearest_band(752.99)
i760 = nearest_band(760.45)
i775 = nearest_band(775.37)

print("\nChannels used in the main analysis:")
print(f"Left continuum : {wl[i753]:.3f} nm")
print(f"O₂ channel     : {wl[i760]:.3f} nm")
print(f"Right/reference: {wl[i775]:.3f} nm")


## 4. Figure 1 — L1B radiance spectrum across the O₂ A-band

A representative scene pixel is used to show the strong radiance depression near the EMIT 760.45 nm channel.


In [ ]:
row = 640
col = 621

spec = rad["radiance"].isel(
    downtrack=row,
    crosstrack=col
).values

plot_region = (wl >= 725) & (wl <= 805)

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    wl[plot_region],
    spec[plot_region],
    marker="o",
    linewidth=1.8,
)

ax.axvline(
    wl[i760],
    linestyle="--",
    linewidth=1.5,
    label=f"EMIT O₂ channel ({wl[i760]:.2f} nm)",
)

ax.axvline(
    wl[i775],
    linestyle="--",
    linewidth=1.5,
    label=f"Reference channel ({wl[i775]:.2f} nm)",
)

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("At-sensor radiance")
ax.set_title(
    "EMIT L1B Radiance Across the O$_2$ A-Band\n"
    "Georgia — 3 June 2026"
)
ax.grid(alpha=0.25)
ax.legend()

fig.savefig(OUT_DIR / "01_radiance_spectrum.png")
plt.show()

L760_pixel = float(spec[i760])
L775_pixel = float(spec[i775])
pixel_index = -np.log(L760_pixel / L775_pixel)

print(f"L760 = {L760_pixel:.6f}")
print(f"L775 = {L775_pixel:.6f}")
print(f"-ln(L760/L775) = {pixel_index:.6f}")


## 5. Figure 2 — Spatial O₂ A-band index

The scene-wide index is

\[
z_{O_2}=-\ln\left(\frac{L_{760}}{L_{775}}\right),
\]

where the nearest EMIT channels are approximately 760.45 nm and 775.37 nm.


In [ ]:
L760 = rad["radiance"].isel(bands=i760).values
L775 = rad["radiance"].isel(bands=i775).values

valid_index = (
    np.isfinite(L760) &
    np.isfinite(L775) &
    (L760 > 0) &
    (L775 > 0)
)

zO2 = np.full(L760.shape, np.nan, dtype=np.float32)

with np.errstate(divide="ignore", invalid="ignore"):
    zO2[valid_index] = -np.log(
        L760[valid_index] / L775[valid_index]
    )

print("Valid pixels:", int(np.sum(valid_index)))
print("Mean O₂ index:", float(np.nanmean(zO2)))
print("SD O₂ index:", float(np.nanstd(zO2)))

vmin, vmax = np.nanpercentile(zO2, [2, 98])

fig, ax = plt.subplots(figsize=(8, 7))

im = ax.imshow(
    zO2,
    vmin=vmin,
    vmax=vmax,
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r"$-\ln(L_{760}/L_{775})$")

ax.set_xlabel("Crosstrack pixel")
ax.set_ylabel("Downtrack pixel")
ax.set_title(
    "EMIT L1B O$_2$ A-Band Index\n"
    "Georgia — 3 June 2026"
)

fig.savefig(OUT_DIR / "02_o2_index_map.png")
plt.show()


## 6. Figure 3 — Continuum-normalized O₂ A-band depth versus elevation

A linear continuum is interpolated between the neighboring 752.99 nm and 775.37 nm channels at the 760.45 nm O₂ channel. The resulting band depth is

\[
D_{O_2}=-\ln\left(\frac{L_{760}}{L_{continuum}}\right).
\]

The quantitative elevation analysis is restricted to **100–2000 m**.


In [ ]:
L753 = rad["radiance"].isel(bands=i753).values

w753 = float(wl[i753])
w760 = float(wl[i760])
w775 = float(wl[i775])

fraction = (w760 - w753) / (w775 - w753)
L_cont = L753 + fraction * (L775 - L753)

with np.errstate(divide="ignore", invalid="ignore"):
    D_O2 = -np.log(L760 / L_cont)

mask = (
    np.isfinite(D_O2) &
    np.isfinite(elev) &
    np.isfinite(L760) &
    np.isfinite(L_cont) &
    (L760 > 0) &
    (L_cont > 0) &
    (elev >= 100) &
    (elev <= 2000)
)

d = D_O2[mask]
h = elev[mask]

r_pixel = np.corrcoef(h, d)[0, 1]
slope_per_m, intercept = np.polyfit(h, d, 1)
slope_per_km = slope_per_m * 1000

# 100-m elevation bins
bins = np.arange(100, 2100, 100)
centers = (bins[:-1] + bins[1:]) / 2

means = []

for lo, hi in zip(bins[:-1], bins[1:]):
    in_bin = (h >= lo) & (h < hi)
    means.append(
        np.mean(d[in_bin]) if np.any(in_bin) else np.nan
    )

means = np.asarray(means)
good_bins = np.isfinite(means)

r_bins = np.corrcoef(
    centers[good_bins],
    means[good_bins]
)[0, 1]

print("Pixels retained:", len(d))
print(f"Pixel correlation: {r_pixel:.6f}")
print(f"100-m-bin correlation: {r_bins:.6f}")
print(f"Slope: {slope_per_km:.6f} km^-1")

fig, ax = plt.subplots(figsize=(8, 5.5))

ax.plot(
    centers,
    means,
    "o-",
    linewidth=1.8,
    markersize=5,
    label="100-m elevation-bin mean",
)

ax.set_xlabel("Surface elevation (m)")
ax.set_ylabel("Continuum-normalized O$_2$ A-band depth")
ax.set_title(
    "EMIT O$_2$ A-Band Depth vs Surface Elevation\n"
    "Restricted to 100–2000 m"
)

summary = (
    f"Pixel r = {r_pixel:.3f}\n"
    f"Bin-mean r = {r_bins:.3f}\n"
    f"Slope = {slope_per_km:.4f} km$^{{-1}}$\n"
    f"N = {len(d):,} pixels"
)

ax.text(
    0.97,
    0.95,
    summary,
    transform=ax.transAxes,
    horizontalalignment="right",
    verticalalignment="top",
    bbox=dict(
        boxstyle="round",
        facecolor="white",
        alpha=0.85,
    ),
)

ax.grid(alpha=0.25)
ax.legend(loc="lower left")

fig.savefig(OUT_DIR / "03_o2_vs_elevation.png")
plt.show()


## 7. Figure 4 — Spectral control

To test whether elevation sensitivity behaves identically across neighboring channels, the slope of

\[
z_\lambda=-\ln\left(\frac{L_\lambda}{L_{775}}\right)
\]

with elevation is evaluated across nearby EMIT wavelengths. The 775.37 nm reference channel itself is omitted because its ratio to itself is identically zero.


In [ ]:
targets = [
    730.61,
    738.07,
    745.53,
    752.99,
    760.45,
    767.91,
    782.83,
    790.29,
    797.75,
]

spectral_results = []

for target in targets:
    i = nearest_band(target)
    Li = rad["radiance"].isel(bands=i).values

    with np.errstate(divide="ignore", invalid="ignore"):
        Z = -np.log(Li / L775)

    m = (
        np.isfinite(Z) &
        np.isfinite(elev) &
        (Li > 0) &
        (L775 > 0) &
        (elev >= 100) &
        (elev <= 2000)
    )

    zz = Z[m]
    hh = elev[m]

    r = np.corrcoef(zz, hh)[0, 1]
    slope_km = np.polyfit(hh, zz, 1)[0] * 1000

    spectral_results.append(
        (float(wl[i]), float(r), float(slope_km))
    )

print(" wavelength_nm       r       slope_per_km")
for wavelength, r, slope_km in spectral_results:
    print(f"{wavelength:13.3f}  {r:8.4f}  {slope_km:13.5f}")

wave_plot = np.array([x[0] for x in spectral_results])
slope_plot = np.array([x[2] for x in spectral_results])

fig, ax = plt.subplots(figsize=(8, 5.5))

ax.plot(
    wave_plot,
    slope_plot,
    "o-",
    linewidth=1.8,
)

ax.axhline(0, linewidth=1)

ax.axvline(
    wl[i760],
    linestyle="--",
    label=f"O₂ channel ({wl[i760]:.2f} nm)",
)

ax.axvline(
    wl[i775],
    linestyle="--",
    label=f"Reference ({wl[i775]:.2f} nm)",
)

ax.set_xlabel("EMIT wavelength (nm)")
ax.set_ylabel("Elevation slope (index units km$^{-1}$)")
ax.set_title(
    "Spectral Dependence of Elevation Sensitivity\n"
    "EMIT L1B — Restricted to 100–2000 m"
)

ax.grid(alpha=0.25)
ax.legend()

fig.savefig(OUT_DIR / "04_spectral_elevation_sensitivity.png")
plt.show()


## 8. Interpretation

For this scene, the continuum-normalized O₂ A-band depth decreases strongly with surface elevation over the restricted 100–2000 m range. This is consistent with changes in atmospheric column length above higher terrain.

The result should **not** be interpreted as direct orbital geometric-range retrieval. EMIT's approximately 8.5 nm channels are substantially broader than the narrow spectral bands used in the related passive-ranging work, and the orbital measurement can also be affected by surface reflectance, atmospheric state, illumination, and viewing geometry.


In [ ]:
print("FINAL SUMMARY")
print("-" * 50)
print(f"Pixel-level r        = {r_pixel:.3f}")
print(f"100-m-bin r          = {r_bins:.3f}")
print(f"Slope                = {slope_per_km:.4f} km^-1")
print(f"Pixels analyzed      = {len(d):,}")
print(f"O₂ channel           = {wl[i760]:.3f} nm")
print(f"Reference channel    = {wl[i775]:.3f} nm")
